# Import Library

In [ ]:
# Import library 

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OrdinalEncoder 
from sklearn.preprocessing import LabelEncoder   
from sklearn.preprocessing import OneHotEncoder 

from scipy import stats
from scipy.special import factorial

from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_val_predict
from statsmodels.tools.eval_measures import mse, rmse
from sklearn import preprocessing

import warnings

## Style from matplotlib

In [ ]:
# list of available styles in matplotlib
print(plt.style.available)

# uss of style from matplotlib
plt.style.use('ggplot')

## Set option in pandas to display all rows and columns in dataset

In [ ]:
# set pandas display options for rows and columns
pd.set_option("display.max_columns",None)
pd.set_option("display.max_rows", None)

## Ignore library Deprecated warning messages

In [ ]:
# define function
def ignore_warning():  
    return warnings.filterwarnings(action='ignore')

# function call
ignore_warning() 


## Load Dataset

In [ ]:
 # define function
def load_dataset(dataset_url):
     # to read csv data
    return pd.read_csv(dataset_url)
 
# call function load_dataset() and store dataset value in population_dataframe
population_dataframe = load_dataset(r'./estimated_population.csv')
population_dataframe.head()

# EDA (Exploratory Data Analysis)

#### Shape of Dataset

In [ ]:
# Dimensions of dataset
population_dataframe.shape

#### First 10 observation

In [ ]:
# Gives top 10 rows of dataset
population_dataframe.head(10)

#### Last 5 observation

In [ ]:
# Gives bottom 5 rows of dataset
population_dataframe.tail()

#### Random 10 observation

In [ ]:
# Gives random sample data
population_dataframe.sample(10)

#### Function to create dictionary of existing and new columns name i.e {old: new}

In [ ]:
# rename columns name i.e no space should be in columns name
# use of try and except to handle if value list has not appropriate length for column
new_column_name = ['statistic_label','year','age_group','sex','region','unit','value']

def columns_dict(cols):
    new_col = {}
    
    if len(cols) == len(population_dataframe.columns):
        try:
            for i in range(len(cols)):
                new_col[population_dataframe.columns[i]] = cols[i]
        except:
            print("An error occurred while creating the column dictionary.")
    else:
        print(f"Your length of Population Dataframe {len(population_dataframe.columns)} != {len(new_column_name)} with length of New Column ")
    
    return new_col

col_dict = columns_dict(new_column_name)


#### Rename Column names

In [ ]:
# inplace = True set the values permenantly so that we can use new columns name
population_dataframe.rename(columns=col_dict, inplace=True)
population_dataframe.head()

#### Drop column

In [ ]:
# drop column STATISTIC Label and Unit because it
population_dataframe.drop({'statistic_label', 'unit'}, axis=1, inplace=True)
population_dataframe.head()

#### Check data types of columnns

In [ ]:
# Checking the types of data
population_dataframe.dtypes

#### Check is there null value on dataset

In [ ]:
# isna().sum() provide sum of null value in each column in datasets
population_dataframe.isnull().sum()

#### Gives information about dataset

In [ ]:
# info() provide information about datasets
population_dataframe.info()

#### Count total observation in each column

In [ ]:
# count total items in each column
population_dataframe.count()

#### Check Duplicated value

In [ ]:
# count duplicated value in dataset
population_dataframe.duplicated().sum()

#### Describe the stastical value of dataset

In [ ]:
# describe() generate statistical descriptive value of dataset
population_dataframe.describe()

#### Description about object type column in dataset

In [ ]:
# describe(include=object) generate statistical descriptive value for object datatype of dataset
population_dataframe.describe(include=object)

#### Display Unique value in each column

In [ ]:
# It return all unique value of series object
population_dataframe['age_group'].unique()

In [ ]:
# It return all unique value of series object
population_dataframe['year'].unique()

### Before data cleaning 

#### Boxplot to check outliers in dataset

In [ ]:
# This shows alot of outlilers
plt.figure(figsize=(10,8))
sns.boxplot( data = population_dataframe, x='value');

#### Histplot to see distribution of data

In [ ]:
# Histplot explain distribution of data i.e Right skewed distribution  data
plt.figure(figsize=(10,8))
sns.histplot(data=population_dataframe, x='value', kde=True, bins=50);

#### Skew value

In [ ]:
# best skew value should be -3 to +3
pop_skew_value = stats.skew(population_dataframe['value'], axis=0, bias=True)
print('Skew Value',pop_skew_value)

#### Kurtosis value

In [ ]:
# best kurtosis value should be -10 to +10
kurtosis_value = stats.kurtosis(population_dataframe['value'], axis=0, bias=True)
print('kurtosis Value',kurtosis_value)

#### Pairplot to see data from different perspectives

In [ ]:
# plot pairplot
plt.figure(figsize=(10,8))
sns.pairplot(population_dataframe, hue="region");

#### Remove State from observation

In [ ]:
# Remove State observation from region because state is sum of all other 8 region population
new_population_dataframe = population_dataframe.loc[population_dataframe['region'] != 'State']
new_population_dataframe.head()

#### Unique value of region column

In [ ]:
# Unique region 
new_population_dataframe['region'].unique()

#### Unique value of sex column

In [ ]:
# Unique sex 
new_population_dataframe['sex'].unique()

#### Unique value of age_group column

In [ ]:
# Unique age_group
new_population_dataframe['age_group'].unique()

#### Remove Both sexes and All age group from observation

In [ ]:
# Remove Both sexes observation from sex column and All ages observation from age_group column because both sexes is sum of male and female population and all ages are sum of rest of categorical age
new_population_dataframe = new_population_dataframe.loc[(new_population_dataframe['sex'] != 'Both sexes') & (new_population_dataframe['age_group'] != 'All ages')]
new_population_dataframe.reset_index(drop=True, inplace=True)
new_population_dataframe.tail(5)

#### Function to plot boxplot

In [ ]:
# Boxplot helps to identify outliers
def plot_boxplot(data_frame, col_name):
    plt.figure(figsize=(10,8))
    sns.boxplot(data = data_frame, x=col_name);

#### Boxplot for year column

In [ ]:
# call plot_boxplot for year column, Hence there are no outliers
plot_boxplot(new_population_dataframe, 'year')

#### Boxplot for value column

In [ ]:
# call plot_boxplot function for value column, Hence there are outliers
plot_boxplot(new_population_dataframe, 'value')

#### Function to remove outliers

In [ ]:
# Function to remove outliers from datasets by using IQR (Interquartile Range)

def remove_outliers(dataframe, col_name):
    Q1 = dataframe[col_name].quantile(0.25)
    Q3 = dataframe[col_name].quantile(0.75)
    IQR = Q3-Q1 
    lower_limit = Q1 - 1.5*IQR
    upper_limit = Q3 + 1.5*IQR
    print('IQR',IQR)
    print('lower_limit',lower_limit)
    print('upper_limit',upper_limit)
    new_data = dataframe[(dataframe['value'] > lower_limit) & (dataframe['value'] < upper_limit)]
    return new_data
 
# 1st degree to remove outliers    
newdata_no_outliers = remove_outliers(new_population_dataframe, 'value')

In [ ]:
# 2st degree to remove outliers
newdata_no_outliers = remove_outliers(newdata_no_outliers, 'value')

In [ ]:
# 3st degree to remove outliers
newdata_no_outliers = remove_outliers(newdata_no_outliers, 'value')

### After data cleaning

#### Boxplot after removing outliers

In [ ]:
plot_boxplot(newdata_no_outliers, 'value')

#### Heatmap to see the correlation between column

In [ ]:
# correlation = population_dataframe.corr()
plt.figure(figsize=(10,8))
correlation = newdata_no_outliers.corr(numeric_only=True)
sns.heatmap(correlation, annot=True, fmt='.2f', linewidths=2, cmap="BrBG")
correlation


#### Shape of Dataset after cleaning

In [ ]:
newdata_no_outliers.shape

#### Describe new dataset

In [ ]:
newdata_no_outliers.describe()

#### Subplot

In [ ]:
# plot subplots
fig, ax = plt.subplots(figsize=(10,8))
ax.scatter(newdata_no_outliers["year"], newdata_no_outliers["value"])
ax.set_xlabel("Years")
ax.set_ylabel("Value in thousands")
plt.show()

In [ ]:
newdata_no_outliers.describe()

In [ ]:
# Histplot explain distribution of data i.e Symmetric distribution  data
plt.figure(figsize=(10,8))
sns.histplot(data=newdata_no_outliers, x='value', kde=True, bins=50);

#### Calculate Skewness for distribution of data

In [ ]:
# best skew value should be -3 to +3
skew_value = stats.skew(newdata_no_outliers['value'], axis=0, bias=True)
print('Skew Value',skew_value)

#### Calculate kurtosis for distribution of data

In [ ]:
# best kurtosis value should be -10 to +10
kurtosis_value = stats.kurtosis(newdata_no_outliers['value'], axis=0, bias=True)
print('kurtosis Value',kurtosis_value)

In [ ]:
newdata_no_outliers.describe(include=object)

In [ ]:
newdata_no_outliers['age_group'].unique()

#### Pairplot to see distribution of data from different view

In [ ]:
plt.figure(figsize=(10,8));
sns.pairplot(newdata_no_outliers, hue="region");
plt.show()

# Statistical Data Analysis and visualization

### Total population by year of all states from 2011-2023

In [ ]:
# Filter population 
all_age_bt_sex_state_df = population_dataframe.loc[(population_dataframe['sex'] == 'Both sexes') & (population_dataframe['region'] == 'State') & (population_dataframe['age_group'] == 'All ages')].reset_index(drop=True)

In [ ]:
# Statistical value for bar graph year and population
all_age_bt_sex_state_df[['year','value']]

#### Statistical value  for bar graph

In [ ]:
print(f"Max: {all_age_bt_sex_state_df[['year','value']].max()},  Min: {all_age_bt_sex_state_df[['year','value']].min()}, Mean: {all_age_bt_sex_state_df[['year','value']].mean()} ")


### Mean

In [ ]:
population_mean_value = newdata_no_outliers['value'].mean()
print("Mean Value of Population: ", population_mean_value)

### Stander deviation 

In [ ]:
population_std_value = newdata_no_outliers['value'].std()
print("Standard Deviation Value of Population: ", population_std_value)

#### Bar graph

In [ ]:
#Bar graph to compare population by year 2011-2023 
plt.figure(figsize=(10,8))
plt.bar(all_age_bt_sex_state_df['year'], all_age_bt_sex_state_df['value'])
plt.title("Total population of Ireland")
plt.ylabel("Population in thousands")
plt.xlabel("Years")
plt.show()

### Child population age 0-4 years, both sex, all states

In [ ]:

age04_bt_sex_state_df = population_dataframe.loc[(population_dataframe['sex'] == 'Both sexes') & (population_dataframe['region'] == 'State') & (population_dataframe['age_group'] == '0 - 4 years')].reset_index(drop=True)
age04_bt_sex_state_df

#### Line graph to show birth rate

In [ ]:
#Bar graph to compare population by year 2011-2023 
plt.figure(figsize=(10,8))
plt.plot(age04_bt_sex_state_df['year'], age04_bt_sex_state_df['value'], marker='*')
plt.title("Population of child from age 0-4 years")
plt.ylabel("Population in thousands")
plt.xlabel("Years")
plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
plt.show()

# why child birth is decreasing each year -> research

###  Older Adults Population age 85 years and above , both sex, all states

In [ ]:
age85_above_bt_sex_state_df = population_dataframe.loc[(population_dataframe['sex'] == 'Both sexes') & (population_dataframe['region'] == 'State') & (population_dataframe['age_group'] == '85 years and over')].reset_index(drop=True)
age85_above_bt_sex_state_df

#### Line graph 

In [ ]:
#Bar graph to compare population by year 2011-2023 
plt.figure(figsize=(10,8))
plt.plot(age85_above_bt_sex_state_df['year'], age85_above_bt_sex_state_df['value'], marker='1')
plt.title("Population age 80 and above")
plt.ylabel("Population in thousands")
plt.xlabel("Years")
plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
plt.show()

#### Define Class with method get_age_data() to get data frame based on  different age group

In [ ]:
# Define class name PopulationByAgeGroup
class PopulationByAgeGroup:
    
    # __init__() is Constructor: The function that runs whenever we make a new object i.e population_data 
    def __init__(self, population_dataframe):
        self.population_dataframe = population_dataframe
        
        #gives uniques all age group
        self.age_group_only = population_dataframe['age_group'].unique()
        print('All age group: ',self.age_group_only)
        
    def get_age_data(self, start_age, end_age, sex, region):
        age_group_years = self.age_group_only[start_age: end_age]
        print('age_group_years',age_group_years)

        # find either value from different age exist or not 
        has_age_group = population_dataframe['age_group'].isin(age_group_years)
        
        # filter data age lies from '15 - 19 years' to  '60 - 64 years'
        filter_age_group_df = (population_dataframe.loc[has_age_group])
        age_group_df = filter_age_group_df.loc[(filter_age_group_df['sex'] == sex) & (filter_age_group_df['region'] == region)]
        age_group_df = age_group_df.groupby(['year']).sum().reset_index()
        return age_group_df


# Initilization of Class PopulationByAgeGroup
population_data = PopulationByAgeGroup(population_dataframe)

### Working population in ireland age 15-64

In [ ]:
# get dataframe of age group from '15 - 19 years' to  '60 - 64 years'
# Call PopulationByAgeGroup
working_age_data = population_data.get_age_data(start_age = 3, end_age= 13, sex='Both sexes', region='State')
print('Woring population in Ireland by year : \n',working_age_data)


#### Line graph

In [ ]:
# function to plot line chart to show trend of population based on age group
def plot_linegraph(data_frame, title,  marker_icon):
    plt.figure(figsize=(10,8))
    plt.plot(data_frame['year'], data_frame['value'], marker=marker_icon)
    plt.title(title)
    plt.ylabel("Population in thousands")
    plt.xlabel("Years")
    plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
    plt.show()
    
# line plot showing trend of working populatin in ireland by years
plot_linegraph(working_age_data, "Population age group between 15 to 64 which is working age group of Ireland", "o")

### Children population in ireland age range  '0 - 4 years' to '15 - 19 years'

In [ ]:
# get dataframe of age group from '0-4 years' to  '10 - 14 years'
# Call PopulationByAgeGroup
children_age_data = population_data.get_age_data(start_age = 0, end_age= 3, sex='Both sexes', region='State')
print('Children population in Ireland by year : \n',children_age_data)


In [ ]:
# line plot showing trend of working populatin in ireland by years
plot_linegraph(children_age_data, "Population age group between 0 to 14 which is children age group of Ireland", "1")

### Older age population in ireland age range 65 and above

In [ ]:
# Call PopulationByAgeGroup
older_age_data = population_data.get_age_data(start_age = 13, end_age= -1, sex='Both sexes', region='State')
print('Older population in Ireland by year : \n',working_age_data)

### Chart data for Children, Working age and Older age for year 2023

In [ ]:
chart_data_frame = [children_age_data, working_age_data, older_age_data]
chart_data_frame = pd.concat([children_age_data['year'], children_age_data['value'], working_age_data['value'], older_age_data['value']], keys=['year','children','working','older'], axis=1).reset_index(drop=True)
chart_data_frame

In [ ]:
chart_data_frame = chart_data_frame[chart_data_frame['year'] == 2023].reset_index(drop=True)
chart_data_frame

#### Pie chart to show children, older, and working population

In [ ]:
pie_data = chart_data_frame.iloc[:,1:].values.flatten()
years = chart_data_frame.iloc[:, 0].values
plt.figure(figsize=(10,8))
plt.pie(pie_data, labels=['Children', 'Working People', 'Older'], shadow=False, startangle=90, autopct='%1.1f%%', wedgeprops={'edgecolor': '#ffffff'} )
plt.title(f'Ratio of Children,  Working people and Older age people in year {years}')
plt.tight_layout()
plt.show()

## Male and Female population in Each state of Ireland 

In [ ]:
class MaleFemalePopulationAnalysis:
    def __init__(self, population_dataframe, year, age_group = 'All ages'):
        self.population_dataframe = population_dataframe
        self.year = year
        self.age_group = age_group
        
        #filter male and female data based on year
        self.filter_year_d = (population_dataframe['year'] == self.year) & (population_dataframe['age_group'] == self.age_group) & (population_dataframe['sex'] != 'Both sexes') & (population_dataframe['region'] != 'State')
        self.filter_year_data = population_dataframe.loc[self.filter_year_d].reset_index()
        
        # Filter for Male data
        self.male_value = (self.filter_year_data[self.filter_year_data['sex'] == 'Male'])['value'].values

        # Filter for Female data
        self.female_value = (self.filter_year_data[self.filter_year_data['sex'] == 'Female'])['value'].values

        # Get all region
        self.all_region = self.filter_year_data['region'].unique()

        self.gender_dict = {
            'Male': np.array(self.male_value),
            'Female': np.array(self.female_value)
        }
        
        self.male_percentage = []
        self.female_percentage = []
        for i in range(len(self.male_value)):
            sums = self.male_value[i] + self.female_value[i]
            self.male_percentage.append(np.round((self.male_value[i]/sums)*100,2))
            self.female_percentage.append(np.round((self.female_value[i]/sums)*100,2))
  
        
    def get_male_female_stack_bar(self):
           
        bottom = np.zeros(len(self.all_region))
        labels={}
    
        fig, ax = plt.subplots(figsize=(10, 8))
        for gender, value in self.gender_dict.items():
            percentages = self.male_percentage if gender == 'Male' else self.female_percentage
            per_values = [f'{v} \n ({p}%)' for v,p in zip(value, percentages)]
            labels = per_values
            bar_container = ax.bar(self.all_region, value, width=0.8, label=gender, bottom=bottom)
            bottom += value
            ax.bar_label(bar_container, label_type='center', color='w', labels=labels, padding=0)
            
        ax.set_title(f"Male and Female Population of Ireland  in each Region {self.year}")
        ax.legend(loc="upper right")
        plt.xlabel("States")
        plt.ylabel("Populations in Thousands")
        plt.show()
     
    
    def get_male_female_line_graph(self, marker_icon, title="Male and Female Population of Ireland  in each Region"):
        plt.figure(figsize=(10,8))
        for gender, value in self.gender_dict.items():
            plt.plot(self.all_region, value, label= gender, marker=marker_icon)
            
        plt.title(f"{title} {self.year}")
        plt.legend()
        plt.grid(color = 'grey', linestyle = '--', linewidth = 0.3)
        plt.xlabel("States")
        plt.ylabel("Populations in Thousands")
        plt.show()

        
# Initialize call MaleFemalePopulationAnalysis
# Data of 2011
male_female_pop_bar_2011 = MaleFemalePopulationAnalysis(population_dataframe, 2011)  
male_female_pop_line_2011 = MaleFemalePopulationAnalysis(population_dataframe, 2011)

# Data of 2023
male_female_pop_bar_2023 = MaleFemalePopulationAnalysis(population_dataframe, 2023)  
male_female_pop_line_2023 = MaleFemalePopulationAnalysis(population_dataframe, 2023)

# Data of Male and Female age 0-4 years
male_female_04_years = MaleFemalePopulationAnalysis(population_dataframe, 2023, age_group='0 - 4 years')


#### Line chart to show Male and Female population in each state of Ireland in Year 2011

In [ ]:
# Line graph
male_female_pop_line_2011.get_male_female_line_graph(marker_icon='o')
## this graph shows the ratio of males and females in each region (i.e states) of Irland from the year 2011 which indicates that the border, west, mid-west, and
# south-east had had similar population ratio of slightly above 200 thousand each whereas the south-west and mid-east had a population of both sex were around 350 thousand ,
# similarly, the Dublin state had the highest population of male and female i.e above 600 thousand each and midland had the lowest population below 100 thousand. here population
# of males and females are shown in the red and blue line graph respectively.

#### Stacked bar chart to show Male and Female population in each state of Ireland in Year 2011

In [ ]:
# Call method get_male_female_stack_bar() from class MaleFemalePopulationAnalysis

male_female_pop_bar_2011.get_male_female_stack_bar()

#### Line chart to show 0-4 Year of Male and Female population  in each state of Ireland in Year 2023

In [ ]:
male_female_04_years.get_male_female_line_graph(title="0-4 years Age Group Male and Female Population of Ireland", marker_icon='*')
## this line graph illustrates the child population in each state of Ireland in the year 2023 where the male child population is slightly higher than the female population in every state of Ireland in the year 2023. As shown in the graph population of male and female children on the border, west, midwest, and southeast is between 10 - 15 thousand in the southwest this number increased to around 20 thousand and in mid east the population is between 20-25 thousand. the highest population is in doublin which is more than 40 thousand and the lowest is in the state of Midland which is below 10 thousand. which indicate that the birth rate of male is higher than female in every state of Ireland in the year 2023.

#### Stacked bar chart to show Male and Female population in each state of Ireland in Year 2023

In [ ]:
male_female_pop_bar_2023.get_male_female_stack_bar()

In [ ]:
male_female_pop_line_2023.get_male_female_line_graph(marker_icon='*')

#### Encoding data 

In [ ]:
# calling the method from preprocessing class and assign into variable
ord_encoder = OrdinalEncoder()             # Ordinal Data
lab_encoder = LabelEncoder()               # Label Data or Nominal Data
cat_encoder = OneHotEncoder(sparse=False)  # Categorical Data | Nominal

# assign data frame to new variable encoded_data_frame
encoded_data_frame = newdata_no_outliers

In [ ]:
encoded_data_frame['age_group'] = ord_encoder.fit_transform(encoded_data_frame[['age_group']])
encoded_data_frame['sex'] = lab_encoder.fit_transform(encoded_data_frame['sex'])
encoded_data_frame['region'] = lab_encoder.fit_transform(encoded_data_frame['region'])
encoded_data_frame.info()


## Statistic Graph

### Normal Distribution

In [ ]:
# Normal Distribution 
# Mean and Standard Deviation of the population are calculated above

n = np.random.normal(population_mean_value, population_std_value, size=10000)
count, bins, ignored = plt.hist(n, 50, density=True)
plt.title("Normal Distribution of Ireland Population", fontsize=16)
plt.figure(figsize=(10,8))
plt.show()




#### Normal distribution in line graph

In [ ]:
normal_df = stats.norm.pdf(newdata_no_outliers['value'], loc=population_mean_value, scale=population_std_value)
plt.plot(newdata_no_outliers['value'], normal_df, 'bs')
plt.figure(figsize=(10,8))
plt.show()


### Poisson Distribution for Population 

In [ ]:

def plot_poisson_distribution(column_value):
    x=pd.Series(column_value).to_numpy()
    y=np.exp(-column_value.mean()) * np.power(column_value.mean(),x)/factorial(x)
    plt.figure(figsize=(10,8))
    plt.title("Poisson Distribution with Lambda 5", fontsize="xx-large")
    plt.plot(x,y,'bs')
    plt.show()

# call function
plot_poisson_distribution(encoded_data_frame['value'])

#### Describe Encoded dataset

In [ ]:
encoded_data_frame.describe()

### Poisson Distribution for Age group 

In [ ]:
# call poission distribution function
plot_poisson_distribution(encoded_data_frame['age_group'])

# Machine Learning Models

### We have encoded dataframe for labeled i.e Nominal, ordinal i.e Ordered data

In [ ]:
encoded_data_frame.head()

In [ ]:
encoded_data_frame.info()

### Scaling Data i.e Standarization because data has high outliers and data is Gussian distributed

In [ ]:
scalar = StandardScaler() # Creating object of class StandardScaler() from sklearn.preprocessing 
encoded_data_frame = pd.DataFrame(scalar.fit_transform(encoded_data_frame), columns=encoded_data_frame.columns)
encoded_data_frame.shape

In [ ]:
encoded_data_frame.head() # Here after standarization values scaled -1 to 1

In [ ]:
encoded_data_frame.shape

#### Seperating dataset for training and testing

In [ ]:
X = encoded_data_frame.drop(['value'], axis=1)
y = encoded_data_frame['value']

#### Selection of best random_state

In [ ]:
def get_best_random_state(X,y):
    best_random_state = None
    
    best_training_accuracy = 0.0
    best_testing_accuracy = 0.0
    
    random_state_range = [*range(1,101)] # list bwtween 1-100
    
    for rndom_state in random_state_range:
        #Split data
        X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=rndom_state)
    
        #Initializing and training ML model
        Ml_Model = LinearRegression()
        Ml_Model.fit(X_train, y_train)
        
        # train score
        train_score = Ml_Model.score(X_train,y_train)
        
        #test score
        test_score = Ml_Model.score(X_test,y_test)        

        # check if random_state value give best performance
        if (test_score > best_testing_accuracy):
            best_training_accuracy = train_score
            best_testing_accuracy = test_score
            best_random_state = rndom_state

    print(f'Best Training Accuracy: {round(best_training_accuracy*100,3)}%')
    print(f'Best Testing Accuracy: {round(best_testing_accuracy*100,3)}%')
    print(f'Best random state : {best_random_state}')
    return best_random_state
    
best_random_state = get_best_random_state(X,y)   

#### Display training and testing data shape

In [ ]:
# selecting random_state = 78 from above test

#Split data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=best_random_state)
print('X_train:',X_train.shape)
print('X_test:',X_test.shape)
print('y_train:',y_train.shape)
print('y_test:',y_test.shape)

## Applying Machine Learning model

### Linear Regression 

In [ ]:
linear_reg_model = LinearRegression()

# fin an OLS model
linear_reg_model.fit(X_train,y_train) 

In [ ]:
# Make Prediction
y_predict_train = linear_reg_model.predict(X_train)
y_predict_test = linear_reg_model.predict(X_test)

linear_reg_model.predict([[2024,7.0,1,3]])

##### Linear regression score

In [ ]:
print("R-squared of the model in Training set is: {}%\n".format(round(linear_reg_model.score(X_train,y_train)*100,3)))
print("*****Statistical Value for Test Model*****\n")
print("R-squared of the model in Test set is: {}%".format(round(linear_reg_model.score(X_test,y_test)*100,3)))
print("Root mean squared error of the prediction is: {}%".format(round(rmse(y_test,y_predict_test)*100,3)))
print("Mean absolute percentage error of the prediction is: {}%".format(round(np.mean(np.abs((y_test - y_predict_test)/ y_test))*100,3)))

### Ridge Regression  

In [ ]:
### Using GridSearchCV for parameter optimization
ridge_regression = GridSearchCV(Ridge(), param_grid={'alpha': [0.01,0.1,5,42,100]}, verbose=1)
ridge_regression.fit(X_train, y_train)
ridge_value = ridge_regression.best_estimator_

In [ ]:
# Make prediction 
y_ridge_pridct_train = ridge_value.predict(X_train)
y_ridge_pridct_test = ridge_value.predict(X_test)

#### Ridge Regression  Score

In [ ]:
print("R-squared of the model in Training set is: {}%\n".format(round(ridge_value.score(X_train,y_train)*100,3)))
print("*****Statistical Value for Test Model*****\n")
print("R-squared of the model in Test set is: {}%".format(round(ridge_value.score(X_test,y_test)*100,3)))
print("Root mean squared error of the prediction is: {}%".format(round(rmse(y_test,y_ridge_pridct_test)*100,3)))
print("Mean absolute percentage error of the prediction is: {}%".format(round(np.mean(np.abs((y_test - y_ridge_pridct_test)/ y_test))*100,3)))

### Lasso Regression 

In [ ]:
### Using GridSearchCV for parameter optimization
lasso_regression = GridSearchCV(Lasso(), param_grid={'alpha': [0.01,0.1,5,42,100]}, verbose=1)
lasso_regression.fit(X_train, y_train)
lasso_value = lasso_regression.best_estimator_

In [ ]:
# Make prediction 
y_lasso_pridct_train = lasso_value.predict(X_train)
y_lasso_pridct_test = lasso_value.predict(X_test)

#### Lasso Regression Score

In [ ]:
print("R-squared of the model in Training set is: {}%\n".format(round(lasso_value.score(X_train,y_train)*100,3)))
print("*****Statistical Value for Test Model*****\n")
print("R-squared of the model in Test set is: {}%".format(round(lasso_value.score(X_test,y_test)*100,3)))
print("Root mean squared error of the prediction is: {}%".format(round(rmse(y_test,y_lasso_pridct_test)*100,3)))
print("Mean absolute percentage error of the prediction is: {}%".format(round(np.mean(np.abs((y_test - y_lasso_pridct_test)/ y_test))*100,3)))

### Decision Tree Regressor

In [ ]:
# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=get_best_random_state(X,y))

#### Fit data to model for training 

In [ ]:
DTRegressor = DecisionTreeRegressor()
DTRegressor.fit(X_train,y_train)

#### Predict test data

In [ ]:
# Make prediction
y_DT_predict_test = DTRegressor.predict(X_test)

In [ ]:
##### Decision Tress Regressor Score
print("R-squared of the model in Training set is: {}%\n".format(round(DTRegressor.score(X_train,y_train)*100,3)))
print("*****Statistical Value for Test Model*****\n")
print("R-squared of the model in Test set is: {}%".format(round(DTRegressor.score(X_test,y_test)*100,3)))
print("Mean squared error of the prediction is: {}%".format(round(mse(y_test,y_DT_predict_test)*100,3)))
print("Mean absolute percentage error of the prediction is: {}%".format(round(np.mean(np.abs((y_test - y_DT_predict_test)/ y_test))*100,3)))

### Random Forst Regressor

In [ ]:
# Split data for training and testing
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=get_best_random_state(X,y))

In [ ]:
RFRegressor = RandomForestRegressor()
RFRegressor.fit(X_train,y_train)

In [ ]:
# Make prediction
y_RF_predict_test = RFRegressor.predict(X_test)

In [ ]:
##### Decision Tress Regressor Score
print("R-squared of the model in Training set is: {}%\n".format(round(RFRegressor.score(X_train,y_train)*100,3)))
print("*****Statistical Value for Test Model*****\n")
print("R-squared of the model in Test set is: {}%".format(round(RFRegressor.score(X_test,y_test)*100,3)))
# print("R-squared of the model in Test set is: {}%".format(round(r2_score(y_test, y_RF_predict_test)*100,3)))
print("Mean squared error of the prediction is: {}%".format(round(mse(y_test,y_RF_predict_test)*100,3)))
print("Mean absolute percentage error of the prediction is: {}%".format(round(np.mean(np.abs((y_test - y_RF_predict_test)/ y_test))*100,3)))

### ElasticNet

In [ ]:
# Train Model
elastic_net = ElasticNet(alpha=1)
elastic_net.fit(X_train, y_train)

# Calculate the Prediction
y_elastic_predict_train = elastic_net.predict(X_train)
y_elastic_predict_test = elastic_net.predict(X_test)

# Calculate Mean square error 
mean_square_error = np.mean((y_elastic_predict_test- y_test)**2)

# Display Mean Square Error
print("Mean Square Error:", mean_square_error)

#### Display graph for testing and training data

In [ ]:
# Draw a scatter plot
plt.figure(figsize=(10,8))
plt.scatter(y_elastic_predict_train,  
            y_elastic_predict_train - y_train, 
            c = 'black', 
            marker = 'o', 
            s = 35,
            alpha = 0.5,
            label = 'Training data')
plt.scatter(y_elastic_predict_test,  
            y_elastic_predict_test - y_test, 
            c = 'lightgreen', 
            marker = 's', 
            s = 35,
            alpha = 0.7,
            label = 'Test data')

plt.xlabel('Predicted values')
plt.ylabel('Residuals')
plt.legend(loc = 'upper left')
plt.hlines(y = 0, xmin = -10, xmax = 50, lw = 2, color = 'red')
plt.xlim([-10, 50])
plt.tight_layout()

# plt.savefig('./figures/slr_residuals.png', dpi=300)
plt.show()

### change to new dataset i.e Year and Value

In [ ]:
year_value_df = all_age_bt_sex_state_df[['year','value']]
year_value_df

In [ ]:
scalar = StandardScaler() # Creating object of class StandardScaler() from sklearn.preprocessing 
year_value_df = pd.DataFrame(scalar.fit_transform(year_value_df), columns=year_value_df.columns)
year_value_df.head()

### seperate the for X-independent and y-dependent

In [ ]:
X = year_value_df.drop(['value'], axis=1)
y = year_value_df['value']

### to find best random value

In [ ]:
new_best_random_state = get_best_random_state(X,y)  

In [ ]:
#Split data
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2, random_state=best_random_state)
print('X_train:',X_train.shape)
print('X_test:',X_test.shape)
print('y_train:',y_train.shape)
print('y_test:',y_test.shape)

### Linear Regression

In [ ]:
linear_reg_model = LinearRegression()

# fin an OLS model
linear_reg_model.fit(X_train,y_train) 

In [ ]:
# train score
train_score = linear_reg_model.score(X_train,y_train)
        
#test score
test_score = linear_reg_model.score(X_test,y_test) 

print('Training Score:', train_score*100)
print('Test Score:', test_score*100)

In [ ]:
# Make Prediction
y_predict_train = linear_reg_model.predict(X_train)
y_predict_test = linear_reg_model.predict(X_test)

linear_reg_model.predict([[2024]])

In [ ]:
print("R-squared of the model in Training set is: {}%\n".format(round(linear_reg_model.score(X_train,y_train)*100,3)))
print("*****Statistical Value for Test Model*****\n")
print("R-squared of the model in Test set is: {}%".format(round(linear_reg_model.score(X_test,y_test)*100,3)))
print("Root mean squared error of the prediction is: {}%".format(round(rmse(y_test,y_predict_test)*100,3)))
print("Mean absolute percentage error of the prediction is: {}%".format(round(np.mean(np.abs((y_test - y_predict_test)/ y_test))*100,3)))